# 15. 주거·통근 인사이트 분석 — 최종 수정본

기존 `15_analyze_housing_commute_insights_final_v3`의 핵심 분석은 유지하고,
이번 합의사항에 해당하는 부분만 수정한다.

## 유지하는 기존 분석
### 분석 1. 주거비–통근 교환관계
- 근무동별 기준 거주동
- 10만 원 절감당 통근시간/교통비/시간비용 증가
- 실질 순절감액
- 손익분기 주거비 절감액
- 회귀 / Spearman / 10만 원 주거비 구간 비교

### 분석 2. 실질 비용 효율
- A: 실질비용효율
- B: 월세착시
- C: 숨은효율
- D: 종합고부담
- 주거비 순위 vs 총부담 순위

### 분석 3. 근무지 효과
- 3축 판정 업무지구별 거주 후보 집계
- Top 10
- Top 10 중복률
- 업무지구 특화도
- 권역 단위 요약

## 이번에 수정하는 내용
1. `0.3 / 0.5 / 0.7` 시간가치계수 삭제
2. 시간가치 기준을 다음 3개로 변경
   - 최저임금
   - 청년 평균임금
   - 사용자 월소득 환산 시급
3. 주거비·통근시간·교통비를 임의 가중한 `추천점수`와 가중치 민감도 분석 삭제
4. 업무지구 정의는 **규모·방향·시간 3축 판정 결과**만 사용
5. `80만원 이하 / 60분 이하`는 업무지구 정의에서 제거
   - 필요하면 사용자 후보 필터 조건에서만 사용

14_3 유형화 결과는 후보를 결정하는 점수가 아니라 **지역 특성 설명용 메타정보**로만 사용한다.

In [2]:
# =========================================================
# 1. 라이브러리 / 기본 설정
# =========================================================

from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import platform

from IPython.display import display
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"

plt.rcParams["axes.unicode_minus"] = False

WORK_DAYS_PER_MONTH = 21
MONTHLY_WORK_HOURS = 209

# 2026년 프로젝트 최종 기획에 사용 중인 최저임금 기준
MINIMUM_WAGE_PER_HOUR = 10_320

# 청년 평균임금은 프로젝트에서 최종 출처/값을 확정한 뒤 입력
# 예: YOUTH_AVG_MONTHLY_WAGE = 3_150_000
YOUTH_AVG_MONTHLY_WAGE = None

# 실제 서비스에서는 사용자 입력값으로 대체
USER_MONTHLY_INCOME = 3_000_000

# 메인 공통 분석은 최저임금 기준
BASE_TIME_VALUE_PER_HOUR = MINIMUM_WAGE_PER_HOUR

MAX_ONEWAY_MINUTES = 90
BASELINE_NEARBY_MINUTES = 30
MIN_SAVING_FOR_100K_METRIC = 100_000
TOP_K = 10

print("월 출근일수:", WORK_DAYS_PER_MONTH)
print("월 근로시간:", MONTHLY_WORK_HOURS)
print("최저임금 시간가치:", f"{MINIMUM_WAGE_PER_HOUR:,}원/시간")
print("사용자 월소득 예시:", f"{USER_MONTHLY_INCOME:,}원")
if YOUTH_AVG_MONTHLY_WAGE is None:
    print("청년 평균임금 기준값은 아직 입력하지 않았습니다.")

월 출근일수: 21
월 근로시간: 209
최저임금 시간가치: 10,320원/시간
사용자 월소득 예시: 3,000,000원
청년 평균임금 기준값은 아직 입력하지 않았습니다.


## 2. 파일 경로

필수 파일:
1. 14_3 최종 유형화 `dong_typology_final.csv`
2. 근무동–거주동 대중교통 경로 `commute_routes_analysis_ready.csv`
3. 11번 거주동별 통근부담 `commute_burden_by_home_dong.csv`

업무지구 분석은 별도 3축 판정 결과
`business_district_3axis_result.csv`가 있을 때 수행한다.

In [3]:
# =========================================================
# 2. 프로젝트 경로 / 입력 파일
# =========================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    REPO_DIR = CURRENT_DIR.parent
elif (CURRENT_DIR / "notebooks").exists():
    REPO_DIR = CURRENT_DIR
else:
    raise FileNotFoundError(
        "저장소 루트 또는 notebooks 폴더에서 실행하세요.\n"
        f"현재 위치: {CURRENT_DIR}"
    )

WORKSPACE_DIR = REPO_DIR.parent
DATA_DIR = WORKSPACE_DIR / "project_data"
PROCESSED_DIR = DATA_DIR / "processed"

TYPOLOGY_FILE = PROCESSED_DIR / "dong_typology_final.csv"
ROUTE_FILE = PROCESSED_DIR / "commute_routes_analysis_ready.csv"
HOME_COMMUTE_FILE = PROCESSED_DIR / "commute_burden_by_home_dong.csv"

# 업무지구 정의는 이 파일의 3축 판정 결과를 사용
BUSINESS_DISTRICT_3AXIS_FILE = (
    PROCESSED_DIR / "business_district_3axis_result.csv"
)

for label, path in [
    ("14_3 유형화", TYPOLOGY_FILE),
    ("10번 경로", ROUTE_FILE),
    ("11번 통근부담", HOME_COMMUTE_FILE),
]:
    if not path.exists():
        raise FileNotFoundError(f"{label} 파일이 없습니다: {path}")

print("14_3 유형화:", TYPOLOGY_FILE)
print("10번 경로:", ROUTE_FILE)
print("11번 통근부담:", HOME_COMMUTE_FILE)
print("3축 업무지구:", BUSINESS_DISTRICT_3AXIS_FILE)

14_3 유형화: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/dong_typology_final.csv
10번 경로: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_routes_analysis_ready.csv
11번 통근부담: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_burden_by_home_dong.csv
3축 업무지구: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/business_district_3axis_result.csv


In [4]:
# =========================================================
# 3. 공통 함수
# =========================================================

def read_csv_clean(path):
    df = pd.read_csv(
        path,
        encoding="utf-8-sig",
        low_memory=False,
    )
    df.columns = (
        df.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )
    return df


def normalize_admin_code_value(value):
    if pd.isna(value):
        return pd.NA

    text = re.sub(r"\.0$", "", str(value).strip())
    digits = re.sub(r"\D", "", text)

    if len(digits) == 8:
        return digits + "00"

    if len(digits) == 10:
        if digits.startswith("00"):
            return digits[2:] + "00"
        return digits

    return pd.NA


def normalize_code(series):
    return (
        series.map(normalize_admin_code_value)
        .astype("string")
    )


def safe_spearman(x, y):
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    y = pd.to_numeric(pd.Series(y), errors="coerce")

    mask = x.notna() & y.notna()

    if mask.sum() < 3:
        return np.nan, np.nan

    return spearmanr(x[mask], y[mask])


def rank_within_group(df, group_col, value_col, ascending=True):
    return (
        df.groupby(group_col)[value_col]
        .rank(method="min", ascending=ascending)
    )


def weighted_mean(group, value_col, weight_col="출근_이동량"):
    values = pd.to_numeric(group[value_col], errors="coerce")

    if (
        weight_col in group.columns
        and group[weight_col].notna().any()
    ):
        weights = pd.to_numeric(
            group[weight_col],
            errors="coerce",
        ).fillna(0)

        valid = values.notna() & (weights > 0)

        if valid.any() and weights[valid].sum() > 0:
            return float(
                np.average(
                    values[valid],
                    weights=weights[valid],
                )
            )

    return float(values.mean())

## 4. 경로 데이터 + 14_3 유형화 결합

In [5]:
# =========================================================
# 4. 데이터 표준화 / 결합
# =========================================================

routes_raw = read_csv_clean(ROUTE_FILE)
typology_raw = read_csv_clean(TYPOLOGY_FILE)
home_commute_raw = read_csv_clean(HOME_COMMUTE_FILE)

required_route_cols = [
    "OD_KEY",
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "내부통근여부",
    "분석용_편도시간_분",
    "분석용_편도거리_km",
    "분석용_편도요금_원",
]

missing = [c for c in required_route_cols if c not in routes_raw.columns]
if missing:
    raise KeyError(f"10번 최종 CSV에 필요한 컬럼이 없습니다: {missing}")

routes = pd.DataFrame({
    "OD_KEY": routes_raw["OD_KEY"].astype("string"),
    "거주동코드": normalize_code(routes_raw["거주동 코드"]),
    "거주동명": routes_raw["거주동 이름"].astype("string"),
    "근무동코드": normalize_code(routes_raw["근무동 코드"]),
    "근무동명": routes_raw["근무동 이름"].astype("string"),
    "내부통근여부": routes_raw["내부통근여부"],
    "편도통근시간_분": pd.to_numeric(
        routes_raw["분석용_편도시간_분"],
        errors="coerce",
    ),
    "편도통근거리_km": pd.to_numeric(
        routes_raw["분석용_편도거리_km"],
        errors="coerce",
    ),
    "편도교통비_원": pd.to_numeric(
        routes_raw["분석용_편도요금_원"],
        errors="coerce",
    ),
})

for col in [
    "출근_이동량",
    "최종_가중치",
    "목적지_출근비중",
    "누적_출근비중",
    "환승횟수",
    "총도보시간_분",
    "도보시간비중",
    "요금산출방식",
    "경로값_산출방식",
]:
    if col in routes_raw.columns:
        routes[col] = routes_raw[col]

if routes["내부통근여부"].dtype != bool:
    routes["내부통근여부"] = (
        routes["내부통근여부"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )

internal_missing = routes["내부통근여부"].isna()
routes.loc[internal_missing, "내부통근여부"] = (
    routes.loc[internal_missing, "거주동코드"]
    .eq(routes.loc[internal_missing, "근무동코드"])
)
routes["내부통근여부"] = routes["내부통근여부"].astype(bool)

if routes["OD_KEY"].duplicated().any():
    raise ValueError("10번 경로 데이터에 중복 OD_KEY가 있습니다.")

external_fee_missing = (
    (~routes["내부통근여부"])
    & routes["편도교통비_원"].isna()
).sum()

if external_fee_missing > 0:
    raise ValueError(
        f"외부 통근 요금 결측이 남아 있습니다: {external_fee_missing:,}개"
    )

# 14_3 유형화
typology = typology_raw.copy()
typology["거주동코드"] = normalize_code(typology["행정동코드"])

TYPOLOGY_KEEP_CANDIDATES = [
    "거주동코드",
    "시군구명",
    "행정동명",
    "권역",
    "군집",
    "행정동_유형",
    "4사분면_주거통근",
    "4사분면_부담구조",
    "유형화_상태",
    "유형화_데이터부족",
    "표면주거비_결측",
    "표면주거비_원",
    "대표_편도통근시간_분",
    "월통근교통비_원",
    "동일통근권_내부출근비율",
    "목적지_정규화엔트로피",
    "목적지_HHI",
    "청년1인세대_비율",
]

typology = (
    typology[
        [c for c in TYPOLOGY_KEEP_CANDIDATES if c in typology.columns]
    ]
    .drop_duplicates("거주동코드")
)

# 11번 내부통근 요금 proxy
home_fee = home_commute_raw[
    ["거주동 코드", "대표_편도교통비_원"]
].copy()
home_fee["거주동코드"] = normalize_code(home_fee["거주동 코드"])
home_fee["거주동대표_편도교통비_원"] = pd.to_numeric(
    home_fee["대표_편도교통비_원"],
    errors="coerce",
)
home_fee = (
    home_fee[
        ["거주동코드", "거주동대표_편도교통비_원"]
    ]
    .drop_duplicates("거주동코드")
)

analysis = (
    routes
    .merge(
        typology,
        on="거주동코드",
        how="left",
        validate="many_to_one",
    )
    .merge(
        home_fee,
        on="거주동코드",
        how="left",
        validate="many_to_one",
    )
)

if "행정동명" in analysis.columns:
    analysis["행정동명"] = analysis["행정동명"].fillna(analysis["거주동명"])
else:
    analysis["행정동명"] = analysis["거주동명"]

analysis["편도교통비_원_원본"] = analysis["편도교통비_원"]

analysis["교통비_15번산출방식"] = np.where(
    analysis["편도교통비_원"].notna(),
    "10번_OD경로요금",
    pd.NA,
)

internal_fee_mask = (
    analysis["내부통근여부"]
    & analysis["편도교통비_원"].isna()
)

analysis.loc[
    internal_fee_mask,
    "편도교통비_원",
] = analysis.loc[
    internal_fee_mask,
    "거주동대표_편도교통비_원",
]

analysis.loc[
    internal_fee_mask,
    "교통비_15번산출방식",
] = "내부통근_11번거주동대표요금_proxy"

print("경로 행:", f"{len(routes):,}")
print("결합 후:", f"{len(analysis):,}")
print("표면주거비 결측:", f"{analysis['표면주거비_원'].isna().sum():,}")
print("교통비 최종 결측:", f"{analysis['편도교통비_원'].isna().sum():,}")

경로 행: 30,839
결합 후: 30,839
표면주거비 결측: 641
교통비 최종 결측: 0


## 5. 공통 부담 계산

기존 15번의 총부담 구조는 유지하되, `time_value_factor`는 제거한다.

- 월 통근시간 = 편도 통근시간 × 2 × 21일
- 월 교통비 = 편도 교통비 × 2 × 21일
- 월 통근시간비용 = 월 통근시간 × 시간당 시간가치
- 총 주거·통근 부담 = 표면주거비 + 월 교통비 + 월 통근시간비용

메인 분석은 **최저임금 기준**으로 계산한다.

In [6]:
# =========================================================
# 5. 공통 부담 계산 — 별도 계수 없음
# =========================================================

def add_burden_metrics(
    df,
    time_value_per_hour,
    time_value_basis,
    work_days=WORK_DAYS_PER_MONTH,
):
    out = df.copy()

    out["월통근시간_시간"] = (
        out["편도통근시간_분"]
        * 2
        * work_days
        / 60
    )

    out["월교통비_원"] = (
        out["편도교통비_원"]
        * 2
        * work_days
    )

    out["월통근시간비용_원"] = (
        out["월통근시간_시간"]
        * float(time_value_per_hour)
    )

    out["총주거통근부담_원"] = (
        out["표면주거비_원"]
        + out["월교통비_원"]
        + out["월통근시간비용_원"]
    )

    out["주거비비중"] = (
        out["표면주거비_원"]
        / out["총주거통근부담_원"]
    )

    out["통근비용비중"] = (
        (
            out["월교통비_원"]
            + out["월통근시간비용_원"]
        )
        / out["총주거통근부담_원"]
    )

    out["시간가치기준"] = time_value_basis
    out["통근시간가치_시간당원"] = float(time_value_per_hour)

    return out


common = add_burden_metrics(
    analysis,
    time_value_per_hour=BASE_TIME_VALUE_PER_HOUR,
    time_value_basis="최저임금",
)

common = common[
    common["편도통근시간_분"].between(
        1,
        MAX_ONEWAY_MINUTES,
    )
].dropna(
    subset=[
        "표면주거비_원",
        "편도통근시간_분",
        "편도교통비_원",
        "총주거통근부담_원",
    ]
).copy()

print("최종 OD 조합:", f"{len(common):,}")
print("근무동 수:", common["근무동코드"].nunique())
print("거주동 수:", common["거주동코드"].nunique())

최종 OD 조합: 30,171
근무동 수: 427
거주동 수: 420


# 분석 1. 주거비를 줄이면 통근 부담이 얼마나 늘어나는가

근무동별로 따로 비교한다.

기준점은 해당 근무동까지 **30분 이내 거주동들의 중위 표면주거비와 가장 가까운 실제 거주동**으로 둔다.

주의할 점은 `10만 원 절감당` 비율이다. 실행 결과에서 기준보다 몇 천 원만 저렴한 동까지
100,000원을 곱해 환산하면서 수천 분 같은 비정상적으로 큰 평균이 나타났다.

따라서:
- 실질 순절감액/월세 착시 판정은 모든 저렴한 후보에서 계산
- **10만 원 절감당 지표는 실제로 10만 원 이상 절감되는 후보에서만 계산**
- 평균보다 중앙값을 핵심값으로 우선 해석

하도록 수정한다.


In [7]:
# =========================================================
# 7. 분석 1 — 근무동별 기준값
# =========================================================

baseline_rows = []

for work_code, g in common.groupby("근무동코드"):
    near = g[
        g["편도통근시간_분"] <= BASELINE_NEARBY_MINUTES
    ].copy()

    if len(near) == 0:
        continue

    # 30분 이내 거주 후보의 중위 주거비
    baseline_housing = near["표면주거비_원"].median()

    # 중위 주거비와 가장 가까운 실제 동을 기준 대표동으로 선택
    ref_idx = (
        near["표면주거비_원"] - baseline_housing
    ).abs().idxmin()

    ref = near.loc[ref_idx]

    baseline_rows.append({
        "근무동코드": work_code,
        "기준거주동코드": ref["거주동코드"],
        "기준거주동명": ref.get("행정동명", pd.NA),
        "기준표면주거비_원": ref["표면주거비_원"],
        "기준편도통근시간_분": ref["편도통근시간_분"],
        "기준월교통비_원": ref["월교통비_원"],
        "기준월통근시간비용_원": ref["월통근시간비용_원"],
        "기준총부담_원": ref["총주거통근부담_원"],
        "30분이내후보수": len(near),
    })

baseline = pd.DataFrame(baseline_rows)

print("기준값 생성 근무동 수:", len(baseline))
display(baseline.head(20))

기준값 생성 근무동 수: 427


,근무동코드,기준거주동코드,기준거주동명,기준표면주거비_원,기준편도통근시간_분,기준월교통비_원,기준월통근시간비용_원,기준총부담_원,30분이내후보수
0,1111051500,1111058000,교남동,"690,833.3333",24.1000,"63,000.0000","174,098.4000","927,931.7333",31
1,1111053000,1111058000,교남동,"690,833.3333",13.7000,"63,000.0000","98,968.8000","852,802.1333",100
2,1111054000,1111051500,청운효자동,"720,833.3333",13.5000,"58,800.0000","97,524.0000","877,157.3333",15
3,1111055000,1111055000,부암동,"647,875.0000",16.4300,"64,874.5332","118,690.3200","831,439.8532",10
4,1111056000,1138055200,갈현제2동,"627,500.0000",27.4000,"63,000.0000","197,937.6000","888,437.6000",11
5,1111057000,1141062000,홍제제1동,"609,166.6667",11.1000,"65,100.0000","80,186.4000","754,453.0667",6
6,1111058000,1111055000,부암동,"647,875.0000",21.8000,"63,000.0000","157,483.2000","868,358.2000",40
7,1111060000,1111064000,이화동,"655,000.0000",14.9000,"63,000.0000","107,637.6000","825,637.6000",65
8,1111061500,1111058000,교남동,"690,833.3333",20.0000,"65,100.0000","144,480.0000","900,413.3333",98
9,1111063000,1123056000,전농제1동,"657,500.0000",26.4000,"63,000.0000","190,713.6000","911,213.6000",100


In [8]:
# =========================================================
# 8. 분석 1 — 주거비 절감 vs 통근부담 증가
# =========================================================

tradeoff = common.merge(
    baseline,
    on="근무동코드",
    how="inner",
)

tradeoff["주거비절감액_원"] = (
    tradeoff["기준표면주거비_원"]
    - tradeoff["표면주거비_원"]
)

tradeoff["편도통근시간증가_분"] = (
    tradeoff["편도통근시간_분"]
    - tradeoff["기준편도통근시간_분"]
)

tradeoff["월교통비증가_원"] = (
    tradeoff["월교통비_원"]
    - tradeoff["기준월교통비_원"]
)

tradeoff["월통근시간비용증가_원"] = (
    tradeoff["월통근시간비용_원"]
    - tradeoff["기준월통근시간비용_원"]
)

tradeoff["통근부담증가액_원"] = (
    tradeoff["월교통비증가_원"]
    + tradeoff["월통근시간비용증가_원"]
)

tradeoff["실질순절감액_원"] = (
    tradeoff["주거비절감액_원"]
    - tradeoff["통근부담증가액_원"]
)

tradeoff["손익분기주거비절감액_원"] = (
    tradeoff["통근부담증가액_원"]
)

# 기준보다 저렴한 후보 전체
cheaper_all = tradeoff[
    tradeoff["주거비절감액_원"] > 0
].copy()

cheaper_all["절감판정"] = np.where(
    cheaper_all["실질순절감액_원"] > 0,
    "실제 부담 절감",
    "월세 착시",
)

# '10만원 절감당' 비율 분석은 실제 절감액 >= 10만원인 후보만 사용
cheaper = cheaper_all[
    cheaper_all["주거비절감액_원"]
    >= MIN_SAVING_FOR_100K_METRIC
].copy()

scale = (
    100_000
    / cheaper["주거비절감액_원"]
)

cheaper["10만원절감당_통근시간증가_분"] = (
    cheaper["편도통근시간증가_분"]
    * scale
)

cheaper["10만원절감당_월교통비증가_원"] = (
    cheaper["월교통비증가_원"]
    * scale
)

cheaper["10만원절감당_시간비용증가_원"] = (
    cheaper["월통근시간비용증가_원"]
    * scale
)

tradeoff_summary = (
    cheaper.groupby("근무동코드")
    .agg(
        분석후보수=("거주동코드", "count"),
        중앙값_10만원절감당_통근시간증가_분=(
            "10만원절감당_통근시간증가_분",
            "median",
        ),
        평균_10만원절감당_통근시간증가_분=(
            "10만원절감당_통근시간증가_분",
            "mean",
        ),
        중앙값_10만원절감당_월교통비증가_원=(
            "10만원절감당_월교통비증가_원",
            "median",
        ),
        중앙값_10만원절감당_시간비용증가_원=(
            "10만원절감당_시간비용증가_원",
            "median",
        ),
        중앙값_실질순절감액_원=(
            "실질순절감액_원",
            "median",
        ),
        월세착시비율=(
            "절감판정",
            lambda s: (s == "월세 착시").mean(),
        ),
    )
    .reset_index()
)

print(
    "10만원 절감당 비율 계산 대상:",
    f"{len(cheaper):,}개 "
    f"(실제 절감액 {MIN_SAVING_FOR_100K_METRIC:,}원 이상)"
)

display(
    tradeoff_summary.sort_values(
        "중앙값_10만원절감당_통근시간증가_분",
        ascending=False,
    ).head(30)
)


10만원 절감당 비율 계산 대상: 8,533개 (실제 절감액 100,000원 이상)


,근무동코드,분석후보수,중앙값_10만원절감당_통근시간증가_분,평균_10만원절감당_통근시간증가_분,중앙값_10만원절감당_월교통비증가_원,중앙값_10만원절감당_시간비용증가_원,중앙값_실질순절감액_원,월세착시비율
334,1171064200,124,28.2843,28.9184,"8,785.7985","204,325.6044","-206,593.0333",0.9113
240,1154551000,9,27.5821,27.5667,0.0000,"199,253.0149","-110,832.5333",0.5556
140,1135058000,1,25.9843,25.9843,0.0000,"187,710.2362","-92,826.6667",1.0000
122,1130560800,1,25.2886,25.2886,0.0000,"182,684.7785","-102,666.9333",1.0000
158,1138057000,4,25.1317,23.9937,"3,500.0000","181,551.3574","-119,975.8667",0.7500
82,1123060000,10,24.5584,23.8470,0.0000,"177,409.7919","-109,000.8000",0.9000
139,1135057000,1,24.5194,24.5194,"5,895.8861","177,128.1419","-88,714.6333",1.0000
92,1126058000,1,24.4878,24.4878,"6,146.3415","176,899.9024","-85,122.4000",1.0000
341,1171071000,120,24.0545,23.9539,"7,554.2565","173,769.7038","-134,904.5333",0.8500
76,1121587000,39,23.4073,22.5049,"4,800.0000","169,094.4420","-107,032.0000",0.8205


### 분석 1 보조분석 — 산점도, 회귀, 10만 원 주거비 구간

기존 분석 기준대로 인과관계가 아니라 **관계/경향**으로 해석한다.

회귀식은 근무동별 단순 관계를 보기 위한 보조자료이고,
사용자 설명에서는 10만 원 주거비 구간별 평균 통근시간이 더 직관적이다.

In [9]:
# =========================================================
# 9. 분석 1 — 근무동별 주거비/통근시간 관계
# =========================================================

regression_rows = []
housing_band_rows = []

for work_code, g in common.groupby("근무동코드"):
    if len(g) < 10:
        continue

    x = g[["표면주거비_원"]].to_numpy()
    y = g["편도통근시간_분"].to_numpy()

    model = LinearRegression()
    model.fit(x, y)

    # 주거비가 10만원 낮아질 때의 시간 변화
    effect_per_minus_100k = (
        model.coef_[0] * -100_000
    )

    rho, p = safe_spearman(
        g["표면주거비_원"],
        g["편도통근시간_분"],
    )

    regression_rows.append({
        "근무동코드": work_code,
        "후보수": len(g),
        "주거비10만원감소_예상통근시간변화_분": effect_per_minus_100k,
        "회귀_R2": model.score(x, y),
        "Spearman_rho": rho,
        "Spearman_p": p,
    })

    temp = g.copy()
    temp["주거비_10만원구간"] = (
        np.floor(temp["표면주거비_원"] / 100_000)
        * 100_000
    )

    band = (
        temp.groupby("주거비_10만원구간")
        .agg(
            행정동수=("거주동코드", "count"),
            평균편도통근시간_분=("편도통근시간_분", "mean"),
            평균월교통비_원=("월교통비_원", "mean"),
        )
        .reset_index()
    )

    band["근무동코드"] = work_code
    housing_band_rows.append(band)

tradeoff_regression = pd.DataFrame(regression_rows)

housing_band_summary = (
    pd.concat(
        housing_band_rows,
        ignore_index=True,
    )
    if housing_band_rows
    else pd.DataFrame()
)

display(tradeoff_regression.head(30))
display(housing_band_summary.head(30))

,근무동코드,후보수,주거비10만원감소_예상통근시간변화_분,회귀_R2,Spearman_rho,Spearman_p
0,1111051500,49,2.4984,0.1047,-0.4721,0.0006
1,1111053000,401,1.6604,0.0350,-0.2671,0.0000
2,1111054000,22,1.4523,0.0234,-0.2570,0.2483
3,1111055000,12,2.5735,0.1195,-0.5524,0.0625
4,1111056000,15,-1.7376,0.0671,-0.2071,0.4588
5,1111058000,50,0.8725,0.0284,-0.4527,0.0010
6,1111060000,155,0.3528,0.0021,-0.1591,0.0480
7,1111061500,420,1.7216,0.0495,-0.2801,0.0000
8,1111063000,256,2.1178,0.0563,-0.3931,0.0000
9,1111064000,199,1.0909,0.0163,-0.1997,0.0047


,주거비_10만원구간,행정동수,평균편도통근시간_분,평균월교통비_원,근무동코드
0,"400,000.0000",2,27.5000,"63,000.0000",1111051500
1,"500,000.0000",14,34.9214,"63,750.0000",1111051500
2,"600,000.0000",16,25.2438,"63,787.5000",1111051500
3,"700,000.0000",10,24.2160,"63,166.6072",1111051500
4,"800,000.0000",5,28.6800,"65,940.0000",1111051500
5,"900,000.0000",2,19.2500,"60,900.0000",1111051500
6,"400,000.0000",32,47.3656,"68,840.6250",1111053000
7,"500,000.0000",120,43.3008,"67,305.0000",1111053000
8,"600,000.0000",121,38.7769,"65,690.0826",1111053000
9,"700,000.0000",76,36.8039,"67,117.1053",1111053000


# 분석 2. 어떤 지역이 실제로 비용 효율적인가

주거비만 싼 곳과 **총 주거·통근 부담이 낮은 곳**이 같은지 비교한다.

근무동별 중위값을 기준으로:
- A: 저주거비·저총부담 → 실질 비용 효율
- B: 저주거비·고총부담 → 월세 착시
- C: 고주거비·저총부담 → 숨은 효율
- D: 고주거비·고총부담 → 종합 부담 큼

을 분류한다.

In [10]:
# =========================================================
# 10. 분석 2 — 비용 효율 유형 / 효율지수
# =========================================================

efficiency = common.copy()

group_medians = (
    efficiency.groupby("근무동코드")
    .agg(
        근무동_중위주거비_원=("표면주거비_원", "median"),
        근무동_중위총부담_원=("총주거통근부담_원", "median"),
        근무동_최소총부담_원=("총주거통근부담_원", "min"),
    )
    .reset_index()
)

efficiency = efficiency.merge(
    group_medians,
    on="근무동코드",
    how="left",
)

low_housing = (
    efficiency["표면주거비_원"]
    <= efficiency["근무동_중위주거비_원"]
)
low_total = (
    efficiency["총주거통근부담_원"]
    <= efficiency["근무동_중위총부담_원"]
)

efficiency["비용효율유형"] = np.select(
    [
        low_housing & low_total,
        low_housing & ~low_total,
        ~low_housing & low_total,
        ~low_housing & ~low_total,
    ],
    [
        "A_실질비용효율",
        "B_월세착시",
        "C_숨은효율",
        "D_종합고부담",
    ],
)

efficiency["비용효율성지수"] = (
    efficiency["근무동_최소총부담_원"]
    / efficiency["총주거통근부담_원"]
    * 100
)

efficiency["주거비순위"] = rank_within_group(
    efficiency,
    "근무동코드",
    "표면주거비_원",
    ascending=True,
)

efficiency["총부담순위"] = rank_within_group(
    efficiency,
    "근무동코드",
    "총주거통근부담_원",
    ascending=True,
)

# 양수면 총부담 기준으로 순위 상승
efficiency["순위변화"] = (
    efficiency["주거비순위"]
    - efficiency["총부담순위"]
)

print("[비용 효율 유형]")
display(
    efficiency["비용효율유형"]
    .value_counts()
    .to_frame("OD수")
)

print("\n[월세 착시 예시]")
display(
    efficiency[
        efficiency["비용효율유형"] == "B_월세착시"
    ]
    .sort_values(
        "순위변화",
        ascending=True,
    )
    .head(30)
)

print("\n[숨은 효율 예시]")
display(
    efficiency[
        efficiency["비용효율유형"] == "C_숨은효율"
    ]
    .sort_values(
        "순위변화",
        ascending=False,
    )
    .head(30)
)

[비용 효율 유형]


,OD수
비용효율유형,
A_실질비용효율,11318
D_종합고부담,11087
B_월세착시,3897
C_숨은효율,3869



[월세 착시 예시]


,OD_KEY,거주동코드,거주동명,근무동코드,근무동명,내부통근여부,편도통근시간_분,편도통근거리_km,편도교통비_원,출근_이동량,최종_가중치,목적지_출근비중,누적_출근비중,환승횟수,총도보시간_분,도보시간비중,요금산출방식,경로값_산출방식,시군구명,행정동명,권역,군집,행정동_유형,유형화_상태,유형화_데이터부족,표면주거비_결측,표면주거비_원,대표_편도통근시간_분,월통근교통비_원,동일통근권_내부출근비율,목적지_정규화엔트로피,목적지_HHI,청년1인세대_비율,거주동대표_편도교통비_원,편도교통비_원_원본,교통비_15번산출방식,월통근시간_시간,월교통비_원,월통근시간비용_원,총주거통근부담_원,주거비비중,통근비용비중,시간가치기준,통근시간가치_시간당원,근무동_중위주거비_원,근무동_중위총부담_원,근무동_최소총부담_원,비용효율유형,비용효율성지수,주거비순위,총부담순위,순위변화
12140,11320681_11650530,1132068100,쌍문4동,1165053000,서초3동,False,84.2000,25.1500,"1,800.0000","8,117.5400",0.0126,0.0101,0.4738,2.0000,5.5000,0.0653,API_선택경로요금,TMAP_최종경로,도봉구,쌍문제4동,동북권,3.0000,주거절감·장거리통근형,유형화 완료,False,False,"472,916.6667",38.7153,"66,971.6477",0.4732,0.8218,0.0133,5.6600,"1,594.5630","1,800.0000",10번_OD경로요금,58.9400,"75,600.0000","608,260.8000","1,156,777.4667",0.4088,0.5912,최저임금,"10,320.0000","635,833.3333","1,046,331.0667","715,674.0000",B_월세착시,61.8679,19.0000,346.0000,-327.0000
12097,11320680_11650520,1132068000,쌍문3동,1165052000,서초2동,False,89.3000,23.0300,"1,800.0000","2,640.7500",0.0044,0.0035,0.7462,1.0000,6.1000,0.0683,API_선택경로요금,TMAP_최종경로,도봉구,쌍문제3동,동북권,3.0000,주거절감·장거리통근형,유형화 완료,False,False,"477,916.6667",39.6987,"65,840.1676",0.4446,0.8422,0.0106,17.9600,"1,567.6230","1,800.0000",10번_OD경로요금,62.5100,"75,600.0000","645,103.2000","1,198,619.8667",0.3987,0.6013,최저임금,"10,320.0000","641,666.6667","1,067,041.6000","789,946.8000",B_월세착시,65.9047,16.0000,328.0000,-312.0000
10587,11305608_11680640,1130560800,번3동,1168064000,역삼1동,False,85.7000,21.2200,"1,500.0000","9,039.2500",0.0168,0.0135,0.2313,0.0000,11.6000,0.1354,API_선택경로요금,TMAP_최종경로,강북구,번3동,동북권,3.0000,주거절감·장거리통근형,유형화 완료,False,False,"503,333.3333",33.5991,"64,263.5754",0.4712,0.8565,0.0119,4.4100,"1,530.0851","1,500.0000",10번_OD경로요금,59.9900,"63,000.0000","619,096.8000","1,185,430.1333",0.4246,0.5754,최저임금,"10,320.0000","638,750.0000","1,086,142.2000","813,394.1333",B_월세착시,68.6159,40.0000,342.0000,-302.0000
10499,11305603_11680640,1130560300,번2동,1168064000,역삼1동,False,86.0000,22.2000,"1,500.0000","8,712.6100",0.0154,0.0123,0.3432,0.0000,9.5000,0.1105,API_선택경로요금,TMAP_최종경로,강북구,번2동,동북권,3.0000,주거절감·장거리통근형,유형화 완료,False,False,"483,333.3333",31.9879,"65,802.1582",0.4991,0.8264,0.0131,9.4200,"1,566.7181","1,500.0000",10번_OD경로요금,60.2000,"63,000.0000","621,264.0000","1,167,597.3333",0.4140,0.5860,최저임금,"10,320.0000","638,750.0000","1,086,142.2000","813,394.1333",B_월세착시,69.6639,29.0000,320.0000,-291.0000
12600,11350570_11680580,1135057000,월계2동,1168058000,삼성1동,False,76.2000,20.7300,"1,800.0000","10,676.1700",0.0122,0.0098,0.3677,1.0000,10.1000,0.1325,API_선택경로요금,TMAP_최종경로,노원구,월계2동,동북권,3.0000,주거절감·장거리통근형,유형화 완료,False,False,"479,166.6667",37.7865,"72,797.4530",0.4470,0.8478,0.0112,5.8300,"1,733.2727","1,800.0000",10번_OD경로요금,53.3400,"75,600.0000","550,468.8000","1,105,235.4667",0.4335,0.5665,최저임금,"10,320.0000","639,791.6667","1,024,333.0667","715,372.5333",B_월세착시,64.7258,24.0000,314.0000,-290.0000
9827,11290810_11560540,1129081000,석관동,1156054000,여의동,False,88.0000,21.7000,"1,800.0000","30,009.7600",0.0231,0.0185,0.1942,1.0000,5.2000,0.0591,API_선택경로요금,TMAP_최종경로,성북구,석관동,동북권,3.0000,주거절감·장거리통근형,유형화 완료,False,False,"536,250.0000",36.0387,"70,029.7374",0.3655,0.8252,0.0165,15.8100,"1,667.3747","1,800.0000",10번_OD경로요금,61.6000,"75,600.0000","635,712.0000","1,247,562.0000",0.4298,0.5702,최저임금,"10,320.0000","639,166.6667","1,058,686.6667","645,220.9333",B_월세착시,51.7185,72.0000,362.0000,-290.0000
17900,11470580_11680650,1147058000,신월3동,1168065000,역삼2동,False,87.0000,23.7800,"1,800.0000","2,185.8500",0.0065,0.0052,0.7338,1.0000,5.2000,0.0598,API_선택경로요금,TMAP_최종경로,양천구,신월3동,서남권,5.0000,저주거비·지역연계형,유형화 완료,False,False,"440,000.0000",30.1502,"65,840.0492",0.7044,0.7876,0.0174,11.2800,"1,567.6202","1,800.0000",10번_OD경로요금,60.9000,"75,600.0000","628,488.0000","1,144,088.0000",0.3846,0.6154,최저임금,"10,320.0000","641,666.6667","1,062,418.0000","819,138.6667",B_월세착시,71.5975,6.0000,288.0000,-282.0000
13657,11350640_11680650,1135064000,상계2동,1168065000,역삼2동,False,86.2000,22.1600,"1,800.0000","5,798.8000",


[숨은 효율 예시]


,OD_KEY,거주동코드,거주동명,근무동코드,근무동명,내부통근여부,편도통근시간_분,편도통근거리_km,편도교통비_원,출근_이동량,최종_가중치,목적지_출근비중,누적_출근비중,환승횟수,총도보시간_분,도보시간비중,요금산출방식,경로값_산출방식,시군구명,행정동명,권역,군집,행정동_유형,유형화_상태,유형화_데이터부족,표면주거비_결측,표면주거비_원,대표_편도통근시간_분,월통근교통비_원,동일통근권_내부출근비율,목적지_정규화엔트로피,목적지_HHI,청년1인세대_비율,거주동대표_편도교통비_원,편도교통비_원_원본,교통비_15번산출방식,월통근시간_시간,월교통비_원,월통근시간비용_원,총주거통근부담_원,주거비비중,통근비용비중,시간가치기준,통근시간가치_시간당원,근무동_중위주거비_원,근무동_중위총부담_원,근무동_최소총부담_원,비용효율유형,비용효율성지수,주거비순위,총부담순위,순위변화
25174,11650510_11680640,1165051000,서초1동,1168064000,역삼1동,False,12.1000,1.5000,"1,200.0000","79,302.4400",0.0841,0.0675,0.3568,0.0000,7.3000,0.6033,API_선택경로요금,TMAP_최종경로,서초구,서초1동,동남권,6.0000,고주거비·직주근접형,유형화 완료,False,False,"750,000.0000",21.2271,"65,313.6852",0.7221,0.6889,0.0412,21.6900,"1,555.0877","1,200.0000",10번_OD경로요금,8.4700,"50,400.0000","87,410.4000","887,810.4000",0.8448,0.1552,최저임금,"10,320.0000","638,750.0000","1,086,142.2000","813,394.1333",C_숨은효율,91.6180,336.0000,19.0000,317.0000
25173,11650510_11650520,1165051000,서초1동,1165052000,서초2동,False,7.5000,0.7200,"1,200.0000","79,669.6200",0.0845,0.0678,0.2892,0.0000,5.2000,0.6933,API_선택경로요금,TMAP_최종경로,서초구,서초1동,동남권,6.0000,고주거비·직주근접형,유형화 완료,False,False,"750,000.0000",21.2271,"65,313.6852",0.7221,0.6889,0.0412,21.6900,"1,555.0877","1,200.0000",10번_OD경로요금,5.2500,"50,400.0000","54,180.0000","854,580.0000",0.8776,0.1224,최저임금,"10,320.0000","641,666.6667","1,067,041.6000","789,946.8000",C_숨은효율,92.4368,305.0000,9.0000,296.0000
25171,11650510_11650530,1165051000,서초1동,1165053000,서초3동,False,10.6000,1.6300,"1,500.0000","136,679.5000",0.1449,0.1164,0.1164,0.0000,6.2000,0.5849,API_선택경로요금,TMAP_최종경로,서초구,서초1동,동남권,6.0000,고주거비·직주근접형,유형화 완료,False,False,"750,000.0000",21.2271,"65,313.6852",0.7221,0.6889,0.0412,21.6900,"1,555.0877","1,500.0000",10번_OD경로요금,7.4200,"63,000.0000","76,574.4000","889,574.4000",0.8431,0.1569,최저임금,"10,320.0000","635,833.3333","1,046,331.0667","715,674.0000",C_숨은효율,80.4513,334.0000,55.0000,279.0000
408,11110600_11110615,1111060000,가회동,1111061500,종로1.2.3.4가동,False,9.6000,1.0200,"1,200.0000","29,293.8800",0.2135,0.1714,0.1714,0.0000,6.0000,0.6250,API_선택경로요금,TMAP_최종경로,종로구,가회동,도심권,6.0000,고주거비·직주근접형,유형화 완료,False,False,"756,666.6667",21.8770,"60,228.0866",0.6433,0.6684,0.0628,14.1800,"1,434.0021","1,200.0000",10번_OD경로요금,6.7200,"50,400.0000","69,350.4000","876,417.0667",0.8634,0.1366,최저임금,"10,320.0000","638,750.0000","979,815.7667","686,992.8000",C_숨은효율,78.3865,348.0000,73.0000,275.0000
26832,11680670_11680580,1168067000,개포2동,1168058000,삼성1동,False,14.8000,3.2600,"1,200.0000","52,465.2200",0.0344,0.0276,0.2445,0.0000,7.2000,0.4865,API_선택경로요금,TMAP_최종경로,강남구,개포2동,동남권,5.0000,저주거비·지역연계형,유형화 완료,False,False,"750,000.0000",31.1920,"65,838.3082",0.6651,0.7736,0.0189,5.7500,"1,567.5788","1,200.0000",10번_OD경로요금,10.3600,"50,400.0000","106,915.2000","907,315.2000",0.8266,0.1734,최저임금,"10,320.0000","639,791.6667","1,024,333.0667","715,372.5333",C_숨은효율,78.8450,331.0000,58.0000,273.0000
26651,11680655_11680640,1168065500,도곡1동,1168064000,역삼1동,False,16.3000,2.1800,"1,500.0000","101,935.7800",0.1150,0.0921,0.0921,0.0000,12.1000,0.7423,API_선택경로요금,TMAP_최종경로,강남구,도곡1동,동남권,4.0000,근접통근 균형형,유형화 완료,False,False,"695,000.0000",25.8483,"64,768.3880",0.7101,0.7186,0.0295,15.7300,"1,542.1045","1,500.0000",10번_OD경로요금,11.4100,"63,000.0000","117,751.2000","875,751.2000",0.7936,0.2064,최저임금,"10,320.0000","638,750.0000","1,086,142.2000","813,394.1333",C_숨은효율,92.8796,281.0000,9.0000,272.0000
26701,11680656_11680640,1168065600,도곡2동,1168064000,역삼1동,False,19.0000,2.7300,"1,500.0000","70,267.7100",0.0782,0.0625,0.1288,0.0000,13.2000,0.6947,API_선택경로요금,TMAP_최종경로,강남구,도곡2동,동남권,4.0000,근접통근 균형형,유형화 완료,False,False,"735,833.3333",28.2494,"64,787.7928",0.6532,0.7624,0.0215,4.8600,"1,542.5665","1,500.0000",10번_OD경로요금,13.3000,"63,000.0000","137,256.0000","936,089.3333",0.7861,0.2139,최저임금,"10,320.0000","638,750.0000","1,086,142.2000","813,394.1333",C_숨은효율,86.8928,321.0000,51.0000,270.0000
25855,11650621_11650530,1165062100,방배4동,1165053000,서초3동,False,13.3000,3.3600,"1,500.0000","56,612.4600",0.0687,0.0551,0.1283,1.0000,3.200

In [11]:
# =========================================================
# 11. 분석 2 — 주거비 순위 vs 총부담 순위 상관
# =========================================================

rank_corr_rows = []

for work_code, g in efficiency.groupby("근무동코드"):
    rho, p = safe_spearman(
        g["주거비순위"],
        g["총부담순위"],
    )

    rank_corr_rows.append({
        "근무동코드": work_code,
        "후보수": len(g),
        "주거비_총부담_순위Spearman": rho,
        "p_value": p,
        "평균절대순위변화": (
            g["순위변화"].abs().mean()
        ),
        "월세착시비율": (
            g["비용효율유형"] == "B_월세착시"
        ).mean(),
        "숨은효율비율": (
            g["비용효율유형"] == "C_숨은효율"
        ).mean(),
    })

rank_correlation = pd.DataFrame(rank_corr_rows)

display(
    rank_correlation.sort_values(
        "주거비_총부담_순위Spearman"
    ).head(30)
)

,근무동코드,후보수,주거비_총부담_순위Spearman,p_value,평균절대순위변화,월세착시비율,숨은효율비율
181,1138055100,15,0.2234,0.4235,3.8000,0.2000,0.2000
130,1129078000,16,0.2676,0.3163,4.6250,0.1875,0.1875
315,1159064000,6,0.3479,0.4993,1.1667,0.1667,0.0000
74,1121583000,29,0.3591,0.0557,7.1034,0.2069,0.2069
86,1123060000,60,0.3907,0.0020,14.7000,0.1500,0.1500
61,1120067000,94,0.3983,0.0001,23.8191,0.1702,0.1702
316,1159065000,9,0.4000,0.2861,1.7778,0.1111,0.1111
182,1138055200,17,0.4034,0.1083,4.0588,0.1765,0.1765
399,1171064200,299,0.4059,0.0000,74.6254,0.2007,0.2007
102,1126058000,21,0.4242,0.0553,4.9048,0.1905,0.1905


### 분석 2 보조 — 시간가치 기준별 비교

기존의 `저소득×0.3 / 중간소득×0.5 / 고소득×0.7` 방식은 삭제한다.

동일한 OD에 대해 아래 시간가치 기준을 **별도 계수 없이 그대로 적용**한다.

- 최저임금
- 청년 평균임금 환산 시급
- 사용자 월소득 환산 시급

청년 평균임금 값이 아직 확정되지 않았다면 해당 시나리오만 자동으로 건너뛴다.

In [12]:
# =========================================================
# 12. 분석 2 — 시간가치 기준별 비용 효율 순위
# =========================================================

TIME_VALUE_SCENARIOS = {
    "최저임금": MINIMUM_WAGE_PER_HOUR,
    "사용자소득": USER_MONTHLY_INCOME / MONTHLY_WORK_HOURS,
}

if YOUTH_AVG_MONTHLY_WAGE is not None:
    TIME_VALUE_SCENARIOS["청년평균임금"] = (
        YOUTH_AVG_MONTHLY_WAGE / MONTHLY_WORK_HOURS
    )

scenario_frames = []

for scenario_name, hourly_value in TIME_VALUE_SCENARIOS.items():
    temp = add_burden_metrics(
        analysis,
        time_value_per_hour=hourly_value,
        time_value_basis=scenario_name,
    )

    temp = temp[
        temp["편도통근시간_분"] <= MAX_ONEWAY_MINUTES
    ].dropna(
        subset=[
            "표면주거비_원",
            "편도교통비_원",
            "총주거통근부담_원",
        ]
    ).copy()

    temp["시나리오"] = scenario_name

    temp["총부담순위"] = rank_within_group(
        temp,
        "근무동코드",
        "총주거통근부담_원",
        ascending=True,
    )

    scenario_frames.append(temp)

scenario_results = pd.concat(
    scenario_frames,
    ignore_index=True,
)

scenario_top10 = (
    scenario_results[
        scenario_results["총부담순위"] <= TOP_K
    ]
    .sort_values(
        ["시나리오", "근무동코드", "총부담순위"]
    )
)

display(scenario_top10.head(50))

,OD_KEY,거주동코드,거주동명,근무동코드,근무동명,내부통근여부,편도통근시간_분,편도통근거리_km,편도교통비_원,출근_이동량,최종_가중치,목적지_출근비중,누적_출근비중,환승횟수,총도보시간_분,도보시간비중,요금산출방식,경로값_산출방식,시군구명,행정동명,권역,군집,행정동_유형,유형화_상태,유형화_데이터부족,표면주거비_결측,표면주거비_원,대표_편도통근시간_분,월통근교통비_원,동일통근권_내부출근비율,목적지_정규화엔트로피,목적지_HHI,청년1인세대_비율,거주동대표_편도교통비_원,편도교통비_원_원본,교통비_15번산출방식,월통근시간_시간,월교통비_원,월통근시간비용_원,총주거통근부담_원,주거비비중,통근비용비중,시간가치기준,통근시간가치_시간당원,시나리오,총부담순위
38570,11290580_11110515,1129058000,돈암1동,1111051500,청운효자동,False,29.2000,7.6500,"1,500.0000","1,930.3000",0.0032,0.0026,0.7961,1.0000,9.9000,0.3390,API_선택경로요금,TMAP_최종경로,성북구,돈암제1동,동북권,2.0000,광역분산 통근형,유형화 완료,False,False,"427,500.0000",28.8100,"64,575.2066",0.3755,0.8208,0.0139,7.4800,"1,537.5049","1,500.0000",10번_OD경로요금,20.4400,"63,000.0000","293,397.1292","783,897.1292",0.5454,0.4546,사용자소득,"14,354.0670",사용자소득,1.0000
30934,11110680_11110515,1111068000,창신2동,1111051500,청운효자동,False,25.8000,4.5100,"1,500.0000","1,880.7900",0.0065,0.0052,0.7128,1.0000,12.2000,0.4729,API_선택경로요금,TMAP_최종경로,종로구,창신제2동,도심권,4.0000,근접통근 균형형,유형화 완료,False,False,"489,458.3333",23.5905,"62,792.2956",0.5142,0.7555,0.0283,10.0000,"1,495.0547","1,500.0000",10번_OD경로요금,18.0600,"63,000.0000","259,234.4498","811,692.7831",0.6030,0.3970,사용자소득,"14,354.0670",사용자소득,2.0000
30480,11110570_11110515,1111057000,무악동,1111051500,청운효자동,False,19.9000,4.2300,"1,500.0000","2,763.9800",0.0155,0.0124,0.5488,1.0000,7.5000,0.3769,API_선택경로요금,TMAP_최종경로,종로구,무악동,도심권,4.0000,근접통근 균형형,유형화 완료,False,False,"562,500.0000",22.9649,"64,543.3468",0.6613,0.7437,0.0302,3.3400,"1,536.7464","1,500.0000",10번_OD경로요금,13.9300,"63,000.0000","199,952.1531","825,452.1531",0.6814,0.3186,사용자소득,"14,354.0670",사용자소득,3.0000
30320,11110550_11110515,1111055000,부암동,1111051500,청운효자동,False,11.6000,1.5300,"1,500.0000","17,230.1900",0.0475,0.0381,0.3202,0.0000,8.6000,0.7414,API_선택경로요금,TMAP_최종경로,종로구,부암동,도심권,5.0000,저주거비·지역연계형,유형화 완료,False,False,"647,875.0000",32.5494,"64,874.5332",0.6474,0.7651,0.0245,13.6100,"1,544.6317","1,500.0000",10번_OD경로요금,8.1200,"63,000.0000","116,555.0239","827,430.0239",0.7830,0.2170,사용자소득,"14,354.0670",사용자소득,4.0000
45827,11410620_11110515,1141062000,홍제1동,1111051500,청운효자동,False,17.4000,4.8500,"1,550.0000","8,769.8300",0.0079,0.0063,0.6106,1.0000,8.3000,0.4770,API_선택경로요금,TMAP_최종경로,서대문구,홍제제1동,서북권,5.0000,저주거비·지역연계형,유형화 완료,False,False,"609,166.6667",26.8024,"69,097.3323",0.6364,0.7847,0.0181,12.2200,"1,645.1746","1,550.0000",10번_OD경로요금,12.1800,"65,100.0000","174,832.5359","849,099.2026",0.7174,0.2826,사용자소득,"14,354.0670",사용자소득,5.0000
38875,11290620_11110515,1129062000,정릉1동,1111051500,청운효자동,False,26.8000,7.1800,"1,500.0000","3,498.8600",0.0062,0.0050,0.6447,0.0000,7.3000,0.2724,API_선택경로요금,TMAP_최종경로,성북구,정릉제1동,동북권,2.0000,광역분산 통근형,유형화 완료,False,False,"525,833.3333",30.4062,"64,436.3928",0.3847,0.8238,0.0137,10.8300,"1,534.1998","1,500.0000",10번_OD경로요금,18.7600,"63,000.0000","269,282.2967","858,115.6300",0.6128,0.3872,사용자소득,"14,354.0670",사용자소득,6.0000
45521,11410520_11110515,1141052000,천연동,1111051500,청운효자동,False,16.9000,2.5000,"1,500.0000","4,624.5300",0.0069,0.0055,0.7226,1.0000,12.3000,0.7278,API_선택경로요금,TMAP_최종경로,서대문구,천연동,서북권,4.0000,근접통근 균형형,유형화 완료,False,False,"627,500.0000",24.4663,"62,932.1287",0.6707,0.7501,0.0257,10.4900,"1,498.3840","1,500.0000",10번_OD경로요금,11.8300,"63,000.0000","169,808.6124","860,308.6124",0.7294,0.2706,사용자소득,"14,354.0670",사용자소득,7.0000
30780,11110650_11110515,1111065000,혜화동,1111051500,청운효자동,False,17.2000,4.5800,"1,500.0000","5,814.6200",0.0095,0.0076,0.6105,0.0000,4.7000,0.2733,API_선택경로요금,TMAP_최종경로,종로구,혜화동,도심권,2.0000,광역분산 통근형,유형화 완료,False,False,"627,083.3333",24.8816,"62,405.9072",0.3471,0.7332,0.0364,41.3200,"1,485.8549","1,500.0000",10번_OD경로요금,12.0400,"63,000.0000","172,822.9665","862,906.2998",0.7267,0.2733,사용자소득,"14,354.0670",사용자소득,8.0000
30877,11110670_11110515,1111067000,창신1동,1111051500,청운효자동,False,27.6000,8.3700,"1,500.0000",831.5600,0.0051,0.0041,0.7429,2.0000,3.6000,0.1304,API_선택경로요금,TMAP_최종경로,종로구,창신제1동,도심권,4.0000,근접통근 균형형,유형화 완료,False,False,"552,500.0000",23.0692,"62,764.2136",0.5449,0.7502,0.0254,14.5100,"1,494.3860","1,

# 분석 3. 근무지에 따라 비용효율적인 거주 후보가 어떻게 달라지는가

업무지구 자체는 `80만원 이하`, `60분 이하`로 정의하지 않는다.

먼저 **규모·방향·시간 3축 판정 결과**에서 최종 업무지구를 가져오고,
그 업무지구에 연결된 주요 OD를 기준으로 거주 후보를 비교한다.

현재 경로 데이터는 거주동별 출근량 누적 80%의 주요 목적지 OD이므로,
업무지구 분석은 서울 427×427 완전조합이 아니라 **확보된 주요 통근 OD 범위**에서 해석한다.

업무지구별 Top10은 임의 가중 추천점수가 아니라,
최저임금 기준 `총주거통근부담_원`이 낮은 순서로 정렬한다.

In [13]:
# =========================================================
# 13. 분석 3 — 3축 판정 업무지구 불러오기
# =========================================================

business_districts = pd.DataFrame()
district_available = {}

if BUSINESS_DISTRICT_3AXIS_FILE.exists():

    business_districts = read_csv_clean(
        BUSINESS_DISTRICT_3AXIS_FILE
    )

    code_col = next(
        (
            c for c in [
                "행정동코드",
                "행정동코드10",
                "근무동코드",
            ]
            if c in business_districts.columns
        ),
        None,
    )

    flag_col = next(
        (
            c for c in [
                "업무지구_판정",
                "업무지구여부",
                "최종업무지구",
            ]
            if c in business_districts.columns
        ),
        None,
    )

    name_col = next(
        (
            c for c in [
                "업무지구명",
                "업무지구",
                "권역명",
                "행정동명",
            ]
            if c in business_districts.columns
        ),
        None,
    )

    if code_col is None or flag_col is None:
        raise KeyError(
            "3축 업무지구 결과에는 행정동코드와 "
            "업무지구 최종 판정 컬럼이 필요합니다."
        )

    business_districts["근무동코드"] = normalize_code(
        business_districts[code_col]
    )

    flag = business_districts[flag_col]

    if flag.dtype != bool:
        flag = (
            flag.astype("string")
            .str.strip()
            .str.lower()
            .map({
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "y": True,
                "n": False,
                "예": True,
                "아니오": False,
            })
        )

    business_districts["업무지구_판정_bool"] = flag.fillna(False)

    selected_bd = business_districts[
        business_districts["업무지구_판정_bool"]
    ].copy()

    if name_col is None:
        selected_bd["업무지구명_최종"] = selected_bd["근무동코드"]
    else:
        selected_bd["업무지구명_최종"] = (
            selected_bd[name_col]
            .astype("string")
            .fillna(selected_bd["근무동코드"])
        )

    district_available = (
        selected_bd
        .groupby("업무지구명_최종")["근무동코드"]
        .apply(lambda s: sorted(set(s.dropna())))
        .to_dict()
    )

    print("[3축 판정 최종 업무지구]")
    for district, codes in district_available.items():
        print(f"- {district}: {len(codes)}개 업무동")

else:
    print(
        "business_district_3axis_result.csv가 없어 "
        "업무지구 단위 분석은 자동으로 건너뜁니다."
    )

business_district_3axis_result.csv가 없어 업무지구 단위 분석은 자동으로 건너뜁니다.


In [14]:
# =========================================================
# 14. 분석 3 — 업무지구별 거주 후보 집계 / Top 10
# =========================================================

district_frames = []
diagnostic_rows = []

for district, work_codes in district_available.items():

    sub = common[
        common["근무동코드"].isin(work_codes)
    ].copy()

    original_n = len(sub)

    # 업무지구 정의와 사용자 후보 필터를 섞지 않는다.
    # 여기서는 확보된 주요 OD 전체를 집계한다.
    filtered = sub.copy()
    filtered_n = len(filtered)

    if filtered_n == 0:
        diagnostic_rows.append({
            "업무지구": district,
            "매칭업무동수": len(work_codes),
            "원본OD수": original_n,
            "분석OD수": 0,
            "최종거주후보수": 0,
        })
        continue

    rows = []

    for home_code, g in filtered.groupby("거주동코드"):
        first = g.iloc[0]

        row = {
            "거주동코드": home_code,
            "업무지구": district,
            "표면주거비_원": first["표면주거비_원"],
            "편도통근시간_분": weighted_mean(
                g,
                "편도통근시간_분",
            ),
            "월교통비_원": weighted_mean(
                g,
                "월교통비_원",
            ),
            "월통근시간비용_원": weighted_mean(
                g,
                "월통근시간비용_원",
            ),
            "총주거통근부담_원": weighted_mean(
                g,
                "총주거통근부담_원",
            ),
            "업무동포착수": g["근무동코드"].nunique(),
        }

        for col in [
            "거주동명",
            "행정동명",
            "시군구명",
            "권역",
            "행정동_유형",
            "4사분면_주거통근",
            "4사분면_부담구조",
            "청년1인세대_비율",
            "유형화_데이터부족",
        ]:
            if col in g.columns:
                row[col] = first[col]

        rows.append(row)

    agg = pd.DataFrame(rows)

    if len(agg):
        # '추천점수'이 아니라 실제 비용 기준 순위
        agg["비용기준순위"] = (
            agg["총주거통근부담_원"]
            .rank(method="min", ascending=True)
        )

        # 기존 후속 분석 코드와 호환하기 위한 별칭
        agg["추천순위"] = agg["비용기준순위"]

        district_frames.append(agg)

    diagnostic_rows.append({
        "업무지구": district,
        "매칭업무동수": len(work_codes),
        "원본OD수": original_n,
        "분석OD수": filtered_n,
        "최종거주후보수": len(agg),
    })

district_diagnostics = pd.DataFrame(diagnostic_rows)

business_district_results = (
    pd.concat(
        district_frames,
        ignore_index=True,
    )
    if district_frames
    else pd.DataFrame()
)

if len(business_district_results):
    business_top10 = (
        business_district_results[
            business_district_results["비용기준순위"] <= TOP_K
        ]
        .sort_values(
            ["업무지구", "비용기준순위"]
        )
        .reset_index(drop=True)
    )

    print("[업무지구별 비용 기준 Top 10]")
    display(business_top10)
else:
    business_top10 = pd.DataFrame()
    print("업무지구 분석 결과가 없습니다.")

display(district_diagnostics)

업무지구 분석 결과가 없습니다.


""


In [15]:
# =========================================================
# 15. 분석 3 — Top10 중복률 / 비용 기준 1순위 비교
# =========================================================

OVERLAP_COLUMNS = [
    "업무지구1",
    "업무지구2",
    "업무지구1_Top후보수",
    "업무지구2_Top후보수",
    "공통동수",
    "Top10_중복률",
    "공통거주동코드",
]

overlap_rows = []

if len(business_top10):

    districts = sorted(
        business_top10["업무지구"]
        .dropna()
        .unique()
    )

    print("비교 가능한 업무지구:", districts)

    for i in range(len(districts)):
        for j in range(i + 1, len(districts)):

            d1 = districts[i]
            d2 = districts[j]

            top1 = (
                business_top10[
                    business_top10["업무지구"] == d1
                ]
                .sort_values("추천순위")
                ["거주동코드"]
                .head(TOP_K)
                .tolist()
            )

            top2 = (
                business_top10[
                    business_top10["업무지구"] == d2
                ]
                .sort_values("추천순위")
                ["거주동코드"]
                .head(TOP_K)
                .tolist()
            )

            common_codes = set(top1) & set(top2)

            denominator = min(
                TOP_K,
                len(top1),
                len(top2),
            )

            overlap_rows.append({
                "업무지구1": d1,
                "업무지구2": d2,
                "업무지구1_Top후보수": len(top1),
                "업무지구2_Top후보수": len(top2),
                "공통동수": len(common_codes),
                "Top10_중복률": (
                    len(common_codes) / denominator
                    if denominator > 0
                    else np.nan
                ),
                "공통거주동코드": " | ".join(
                    sorted(common_codes)
                ),
            })


district_top10_overlap = pd.DataFrame(
    overlap_rows,
    columns=OVERLAP_COLUMNS,
)

if len(district_top10_overlap):
    display(
        district_top10_overlap.sort_values(
            "Top10_중복률"
        )
    )
else:
    print(
        "업무지구 간 Top10 중복률을 계산할 수 없습니다."
    )


# 업무지구별 비용 기준 1순위
if len(business_top10):
    district_top1 = (
        business_top10
        .sort_values(["업무지구", "추천순위"])
        .groupby("업무지구")
        .head(1)
        [
            [
                "업무지구",
                "거주동코드",
                "거주동명",
                "행정동_유형",
                "추천순위",
                "총주거통근부담_원",
                "편도통근시간_분",
                "표면주거비_원",
            ]
        ]
        .reset_index(drop=True)
    )

    print("\n[업무지구별 비용 기준 1순위]")
    display(district_top1)
else:
    district_top1 = pd.DataFrame()


업무지구 간 Top10 중복률을 계산할 수 없습니다.


In [16]:
# =========================================================
# 15-1. 업무지구 분석 데이터 품질 확인
# =========================================================

if len(district_diagnostics):
    valid_districts = district_diagnostics[
        district_diagnostics["최종거주후보수"] > 0
    ]

    print(
        "후보가 생성된 업무지구:",
        valid_districts["업무지구"].tolist(),
    )

    print(
        "Top10 비교가 가능한 업무지구 수:",
        int(
            (
                valid_districts["최종거주후보수"]
                >= TOP_K
            ).sum()
        ),
    )

    display(
        valid_districts.sort_values(
            "최종거주후보수",
            ascending=False,
        )
    )


In [17]:
# =========================================================
# 16. 분석 3 — 순위 변화 / 업무지구 특화도
# =========================================================

if len(business_district_results):

    rank_pivot = (
        business_district_results
        .pivot_table(
            index="거주동코드",
            columns="업무지구",
            values="추천순위",
            aggfunc="first",
        )
    )

    burden_pivot = (
        business_district_results
        .pivot_table(
            index="거주동코드",
            columns="업무지구",
            values="총주거통근부담_원",
            aggfunc="first",
        )
    )

    specialization_rows = []

    for district in burden_pivot.columns:

        others = [
            c for c in burden_pivot.columns
            if c != district
        ]

        if not others:
            continue

        other_mean = burden_pivot[others].mean(
            axis=1,
            skipna=True,
        )

        score = (
            other_mean
            - burden_pivot[district]
        )

        valid = (
            burden_pivot[district].notna()
            & other_mean.notna()
        )

        temp = pd.DataFrame({
            "거주동코드": burden_pivot.index[valid],
            "업무지구": district,
            "업무지구특화도_원": score[valid].values,
        })

        specialization_rows.append(temp)

    district_specialization = (
        pd.concat(
            specialization_rows,
            ignore_index=True,
        )
        if specialization_rows
        else pd.DataFrame(
            columns=[
                "거주동코드",
                "업무지구",
                "업무지구특화도_원",
            ]
        )
    )

    if len(district_specialization):
        display(
            district_specialization.sort_values(
                "업무지구특화도_원",
                ascending=False,
            ).head(50)
        )

    print("[업무지구별 순위 피벗]")
    display(rank_pivot.head(30))

else:
    rank_pivot = pd.DataFrame()
    district_specialization = pd.DataFrame(
        columns=[
            "거주동코드",
            "업무지구",
            "업무지구특화도_원",
        ]
    )


### 분석 3 권역 단위 요약

동 단위 추천만 보여주면 인접 동이 연속으로 나올 수 있으므로,
14_2 결과에 `권역`이 있다면 **권역 → 세부 추천동**의 2단계로 요약한다.

권역이 없다면 이 셀은 자동으로 건너뛴다.

In [18]:
# =========================================================
# 17. 분석 3 — 권역 단위 비용 효율 요약
# =========================================================

if (
    len(business_district_results)
    and "권역" in business_district_results.columns
):
    district_region_summary = (
        business_district_results
        .dropna(subset=["권역"])
        .groupby(["업무지구", "권역"])
        .agg(
            후보동수=("거주동코드", "count"),
            평균총부담_원=("총주거통근부담_원", "mean"),
            평균편도통근시간_분=("편도통근시간_분", "mean"),
            평균표면주거비_원=("표면주거비_원", "mean"),
        )
        .reset_index()
    )

    district_region_summary["권역순위"] = (
        district_region_summary
        .groupby("업무지구")["평균총부담_원"]
        .rank(method="min")
    )

    display(
        district_region_summary.sort_values(
            ["업무지구", "권역순위"]
        )
    )
else:
    district_region_summary = pd.DataFrame()
    print("권역 컬럼이 없어 권역 분석은 건너뜁니다.")

권역 컬럼이 없어 권역 분석은 건너뜁니다.


## 분석 4. 사용자 조건별 후보 필터

기존 15번의 임의 가중 추천점수 분석은 삭제한다.

대신 서비스에서 사용자가 입력한:
- 월 주거비 예산
- 허용 가능한 편도 통근시간
- 시간가치 기준

으로 **후보만 필터링**하고, 주거비·통근시간·교통비·시간비용을 각각 보여준다.

따라서 `80만원 / 60분`은 업무지구 정의 기준이 아니라,
필요 시 사용자 입력 예시로만 사용한다.

In [19]:
# =========================================================
# 18. 사용자 조건별 후보 비교 함수
# =========================================================

def compare_home_candidates(
    work_dong_code,
    housing_budget=None,
    max_oneway_minutes=None,
    time_value_basis="사용자소득",
    top_n=20,
):

    if time_value_basis not in TIME_VALUE_SCENARIOS:
        raise ValueError(
            f"사용 가능한 시간가치 기준: {list(TIME_VALUE_SCENARIOS)}"
        )

    temp = add_burden_metrics(
        analysis,
        time_value_per_hour=TIME_VALUE_SCENARIOS[time_value_basis],
        time_value_basis=time_value_basis,
    )

    temp = temp[
        temp["근무동코드"] == str(work_dong_code)
    ].copy()

    if housing_budget is not None:
        temp = temp[
            temp["표면주거비_원"] <= housing_budget
        ]

    if max_oneway_minutes is not None:
        temp = temp[
            temp["편도통근시간_분"] <= max_oneway_minutes
        ]

    temp = temp.dropna(
        subset=[
            "표면주거비_원",
            "편도통근시간_분",
            "편도교통비_원",
            "총주거통근부담_원",
        ]
    )

    temp = temp.sort_values(
        [
            "총주거통근부담_원",
            "편도통근시간_분",
            "표면주거비_원",
        ]
    )

    cols = [
        "근무동코드",
        "근무동명",
        "거주동코드",
        "거주동명",
        "시군구명",
        "권역",
        "행정동_유형",
        "4사분면_주거통근",
        "4사분면_부담구조",
        "표면주거비_원",
        "편도통근시간_분",
        "월교통비_원",
        "월통근시간_시간",
        "월통근시간비용_원",
        "총주거통근부담_원",
        "시간가치기준",
        "통근시간가치_시간당원",
    ]

    return temp[
        [c for c in cols if c in temp.columns]
    ].head(top_n)

# 최종 산출물

이번 수정에서 삭제되는 산출물:
- 임의 가중 `추천점수`
- 주거비 가중치 민감도
- 가중치 전환점
- 추천점수 안정성

그 외 기존 핵심 분석 결과는 유지한다.

In [20]:
# =========================================================
# 19. 최종 결과 저장
# =========================================================

# PROCESSED_DIR.mkdir(
#     parents=True,
#     exist_ok=True,
# )

# OUTPUTS = {
#     "common_table":
#         PROCESSED_DIR / "insight_common_work_home_table.csv",

#     "tradeoff_all_cheaper":
#         PROCESSED_DIR / "insight_housing_commute_tradeoff_all_cheaper.csv",

#     "tradeoff_100k_detail":
#         PROCESSED_DIR / "insight_housing_commute_tradeoff_100k_detail.csv",

#     "tradeoff_summary":
#         PROCESSED_DIR / "insight_housing_commute_tradeoff_summary.csv",

#     "tradeoff_regression":
#         PROCESSED_DIR / "insight_housing_commute_tradeoff_regression.csv",

#     "housing_band":
#         PROCESSED_DIR / "insight_housing_band_commute_summary.csv",

#     "efficiency":
#         PROCESSED_DIR / "insight_cost_efficiency_by_work_home.csv",

#     "rank_correlation":
#         PROCESSED_DIR / "insight_housing_vs_total_rank_correlation.csv",

#     "time_value_top10":
#         PROCESSED_DIR / "insight_time_value_scenario_top10.csv",

#     "business_diagnostics":
#         PROCESSED_DIR / "insight_business_district_diagnostics.csv",

#     "business_top10":
#         PROCESSED_DIR / "insight_business_district_top10.csv",

#     "business_top1":
#         PROCESSED_DIR / "insight_business_district_top1.csv",

#     "business_overlap":
#         PROCESSED_DIR / "insight_business_district_top10_overlap.csv",

#     "business_specialization":
#         PROCESSED_DIR / "insight_business_district_specialization.csv",

#     "business_region":
#         PROCESSED_DIR / "insight_business_district_region_summary.csv",
# }

# SAVE_OBJECTS = {
#     "common_table": common,
#     "tradeoff_all_cheaper": cheaper_all,
#     "tradeoff_100k_detail": cheaper,
#     "tradeoff_summary": tradeoff_summary,
#     "tradeoff_regression": tradeoff_regression,
#     "housing_band": housing_band_summary,
#     "efficiency": efficiency,
#     "rank_correlation": rank_correlation,
#     "time_value_top10": scenario_top10,
#     "business_diagnostics": district_diagnostics,
#     "business_top10": business_top10,
#     "business_top1": district_top1,
#     "business_overlap": district_top10_overlap,
#     "business_specialization": district_specialization,
#     "business_region": district_region_summary,
# }

# for key, path in OUTPUTS.items():
#     df = SAVE_OBJECTS[key]

#     if isinstance(df, pd.DataFrame):
#         df.to_csv(
#             path,
#             index=False,
#             encoding="utf-8-sig",
#         )

# print("[저장 완료]")
# for key, path in OUTPUTS.items():
#     print(f"- {key}: {path}")

## 해석 시 주의

- 주거비와 통근시간 관계는 **인과관계가 아니라 경향**으로 표현한다.
- 총부담은 시간가치 기준에 따라 달라지므로 단일 숫자를 절대값처럼 해석하지 않는다.
- `0.3 / 0.5 / 0.7`과 같은 별도 시간가치계수는 사용하지 않는다.
- 주거비·통근시간·교통비를 임의 가중한 추천점수는 사용하지 않는다.
- 업무지구 정의는 규모·방향·시간 3축 판정 결과를 사용한다.
- 예산·허용 통근시간은 업무지구 정의가 아니라 **사용자 후보 필터**에만 사용한다.
- 유형화 결과는 추천을 직접 결정하지 않고, 후보 지역의 특성을 설명하는 메타정보로 사용한다.
- 현재 경로 데이터는 거주동별 출근량 누적 80% 주요 OD이므로 서울 전체 완전조합으로 해석하지 않는다.

## 서비스 활용을 위한 핵심 인사이트 정리

본 분석은 특정 지역을 일괄적으로 추천하기 위한 것이 아니라, **주거비만으로는 보이지 않는 통근 부담을 사용자에게 직관적으로 보여주기 위한 사전 인사이트 도출**을 목적으로 한다.

### 1. 월세가 저렴하다고 실제 부담까지 낮은 것은 아니다

근무동–거주동 조합을 기준으로 주거비와 통근부담을 함께 비교한 결과,  
주거비만 보면 상대적으로 저렴하지만 **교통비와 통근시간의 시간가치를 포함하면 총부담이 높은 경우**가 확인되었다.

비용효율 유형을 구분한 결과:

- **실질비용효율형:** 11,318개
- **월세착시형:** 3,897개
- **숨은효율형:** 3,869개
- **종합고부담형:** 11,087개

특히 약 3,900개의 `월세착시형` 조합이 나타났다는 점은  
**“월세가 싸다 = 실제 생활부담이 낮다”는 판단이 항상 성립하지 않음**을 보여준다.

> 서비스 메시지 예시  
> **“월세는 저렴한데, 출퇴근까지 생각하면 정말 더 아낄 수 있을까요?”**  
> 주거비는 낮지만 통근시간과 교통비를 포함하면 오히려 전체 부담이 커지는 경우도 있습니다.

---

### 2. 반대로 월세가 조금 높더라도 전체 부담은 더 낮을 수 있다

`숨은효율형` 역시 약 3,900개 조합이 확인되었다.

이는 표면주거비만 보면 상대적으로 비싸 보이는 지역이라도  
통근시간이 짧고 교통비 부담이 낮다면 **주거비와 통근비용을 합친 전체 부담은 오히려 낮아질 수 있음**을 의미한다.

> 서비스 메시지 예시  
> **“월세가 조금 더 비싸도, 가까이 사는 게 실제로는 더 저렴할 수 있어요.”**

따라서 서비스에서는 월세만 비교하기보다  
`주거비 + 교통비 + 통근시간비용`을 함께 보여주는 방식이 적절하다.

---

### 3. 통근시간은 누적하면 생각보다 큰 비용이 된다

편도 통근시간 자체는 하루 단위로 보면 크지 않아 보일 수 있지만,  
월 21일 출근을 기준으로 왕복 통근시간을 누적하면 상당한 시간이 된다.

본 분석에서는 이 시간을 다음과 같이 월 단위로 환산한다.

`월 통근시간 = 편도 통근시간 × 2 × 월 21일`

그리고 해당 시간을 최저임금, 청년 평균임금 또는 사용자 소득 기준의 시간가치로 환산하여  
`월 통근시간비용`으로 보여줄 수 있다.

> 서비스 메시지 예시  
> **“편도 50분이면 한 달에 약 35시간을 출퇴근에 사용합니다.”**  
> 금액으로 환산하면, 월세에는 보이지 않던 통근 부담을 확인할 수 있습니다.

이 값은 실제로 사용자가 지출하는 현금 비용은 아니므로  
`손실액`이나 `지출액`보다는 **“통근에 사용되는 시간의 가치”**로 안내하는 것이 적절하다.

---

### 4. 10만 원 저렴한 집이 반드시 10만 원의 절약을 의미하지 않는다

주거비가 최소 10만 원 이상 저렴한 대안들을 비교한 결과,  
일부 조합에서는 주거비 절감액보다 통근시간비용과 교통비 증가액이 더 크게 나타났다.

즉,

`실질 순절감액 = 주거비 절감액 - 추가 교통비 - 추가 통근시간비용`

으로 보았을 때 실질 순절감액이 작아지거나 음수가 되는 사례가 존재한다.

> 서비스 메시지 예시  
> **“월세 10만 원을 아끼기 위해 통근시간이 크게 늘어난다면?”**  
> 절약한 주거비 일부가 교통비와 통근시간 부담으로 다시 발생할 수 있습니다.

이 결과는 사용자에게 단순히 `싼 지역`을 추천하기보다  
**주거비와 통근 사이의 교환관계(trade-off)를 직접 비교하게 하는 서비스 구조**가 필요함을 보여준다.

---

### 5. 따라서 서비스에서는 하나의 종합 추천점수보다 개별 부담을 직접 보여준다

분석 초기에는 주거비, 통근시간, 교통비에 임의 가중치를 부여한 종합점수를 검토했으나,  
가중치 설정의 근거가 약하고 통근시간이 시간비용에 이미 반영되는 경우 중복 계산 문제가 발생할 수 있어 최종적으로 제외하였다.

대신 서비스에서는 후보 지역별로 다음 값을 직접 비교한다.

- 표면주거비
- 편도 통근시간
- 예상 월 교통비
- 월 통근시간
- 시간가치 기준 월 통근시간비용
- 주거비와 통근비용을 함께 본 전체 부담

이를 통해 사용자가 자신의 우선순위에 따라 직접 판단할 수 있도록 한다.

---

## 서비스 적용 방향

15번 분석 결과는 특정 행정동을 일괄적으로 추천하기 위한 결과가 아니라,  
사용자가 서비스에 들어왔을 때 **“주거비만 보고 있던 관점을 통근 부담까지 확장시키는 콘텐츠”**로 활용한다.

예를 들어 첫 화면이나 결과 대시보드에서 다음과 같은 기사형 인사이트를 제공할 수 있다.

> **월세만 보면 놓치는 비용이 있습니다.**  
> 서울 주요 통근 경로를 분석한 결과, 주거비는 저렴하지만 통근시간과 교통비를 함께 고려하면 전체 부담이 더 높은 경우가 확인되었습니다.

> **매일의 출퇴근, 한 달로 모으면 얼마나 될까요?**  
> 편도 통근시간을 월 단위로 환산하면 생각보다 많은 시간을 이동에 사용하고 있을 수 있습니다.

> **조금 비싼 집이 오히려 더 효율적일 수도 있습니다.**  
> 직장과 가까워 통근시간과 교통비가 크게 줄어드는 경우, 월 전체 부담은 더 낮아질 수 있습니다.

따라서 15번 분석의 핵심 역할은 **사용자에게 정답 지역을 제시하는 것이 아니라, 현재 주거 선택에서 놓치기 쉬운 통근 부담을 데이터로 체감시키는 것**이다.